# Data Types

Split from `01_variables_and_data_types.ipynb` to keep this topic self-contained.

**Navigation:** [Topic overview](./01_variables_and_data_types.ipynb) · [Previous: Variables](./01_variables.ipynb)

## Why this notebook matters

The five core types — `int`, `float`, `str`, `bool`, `None` — are the building blocks of all Python programs. In bioinformatics, knowing which type to use matters: a sequence length is an `int` (never a `float`), GC content is a `float`, a DNA sequence is a `str`, and a gene annotation that has not been loaded yet should be `None`, not an empty string. Getting types right prevents silent errors when doing arithmetic or writing output files.

---[< Previous: Variables + Data Types: Overview](01_variables_and_data_types.ipynb) | [Tier 1: Python for Bioinformatics Overview](../README.md) | [Next: Operators >](../03_Operators_and_Expressions/01_operators.ipynb)---

## How to use this notebook

1. Run cells in order. Sections 3–7 each cover one type; Section 8 covers conversions between them.
2. The string section is the longest — strings are the most important type for bioinformatics work. Do not skip it.
3. The exercises at the end (Section 10) are designed to be run and verified — try them before looking at the hints.

## Common stumbling points

- **Floating-point arithmetic is not exact:** `0.1 + 0.2` is `0.30000000000000004`, not `0.3`. This is not a Python bug — it affects every language that uses IEEE 754 doubles. Always compare floats with a tolerance (`abs(a - b) < 1e-9`), not `==`.
- **`int()` truncates, it does not round:** `int(3.9)` is `3`. Use `round(3.9)` to get `4`. This distinction matters when converting a read depth fraction to an integer.
- **String immutability:** You cannot write `dna[0] = "G"`. Strings cannot be changed in place. Every "modification" creates a new string: `mutated = "G" + dna[1:]`.
- **`None` vs. empty string:** `None` means "this value does not exist". An empty string `""` means "this value exists but has zero length". These are different and should not be used interchangeably — especially when parsing FASTA or VCF files where a missing field is semantically different from an empty field.
- **`is None` not `== None`:** Always check for None with `is`: `if result is None`. The `==` check can sometimes give surprising results if a class defines custom equality.

# Module 2: Variables and Data Types

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Create variables and follow Python naming conventions
2. Work with all basic data types: `int`, `float`, `str`, `bool`, `None`
3. Convert between types safely
4. Use strings to represent and manipulate biological sequences
5. Apply these skills to real bioinformatics calculations

---

## 1. Variables

A **variable** is a name that refers to a value stored in memory. In Python you create a variable simply by assigning a value to a name.

```
variable_name = value
```

Think of it as labeling a container:

```
gene_name -----> "BRCA1"     (the label points to the value)
seq_length ----> 5590        (another label, another value)
```

In [ ]:
# Assigning values to variables
gene_name = "BRCA1"
chromosome = 17
gc_content = 0.423
is_tumor_suppressor = True

print(gene_name)
print(chromosome)
print(gc_content)
print(is_tumor_suppressor)

### Naming rules and conventions

**Rules** (breaking these causes an error):
- Names can contain letters, digits, and underscores
- Names must start with a letter or underscore (not a digit)
- Names are case-sensitive (`gene` and `Gene` are different variables)
- Python keywords (`if`, `for`, `class`, ...) cannot be used as names

**Conventions** (follow these for readable code):
- Use `snake_case` for variable and function names: `gene_name`, `sequence_length`
- Use `UPPER_CASE` for constants: `AVOGADRO = 6.022e23`
- Choose descriptive names: `gc_content` is better than `gc` or `x`

In [ ]:
# Good variable names
sequence_length = 1500
gene_name = "TP53"
melting_temperature = 65.5

# Bad variable names (but technically valid)
x = 1500              # meaningless
seqLen = 1500         # not snake_case (this is camelCase)
three = 1             # misleading!

# This will cause a SyntaxError:
# 2nd_gene = "EGFR"  # cannot start with a digit

### Multiple assignment

Python allows assigning multiple variables in one line. This is especially useful when unpacking related values.

In [ ]:
# Assign multiple variables at once
a_count, t_count, g_count, c_count = 250, 245, 280, 275
print(f"A: {a_count}, T: {t_count}, G: {g_count}, C: {c_count}")

# Swap two variables (elegant Python idiom)
sense_strand = "ATGCGA"
antisense_strand = "TACGCT"

sense_strand, antisense_strand = antisense_strand, sense_strand
print(f"After swap -- sense: {sense_strand}, antisense: {antisense_strand}")

### Variables are references, not boxes

In Python, a variable does not *contain* a value -- it *points to* an object in memory. The built-in `id()` function shows the memory address of an object.

In [ ]:
sequence = "ATGCGATCG"
print(f"id of sequence: {id(sequence)}")

# Reassignment creates a new object (strings are immutable)
sequence = sequence + "AAA"
print(f"id after concatenation: {id(sequence)}")
print(f"New value: {sequence}")

---

## 2. Data Types Overview

Every value in Python has a **type** that determines what operations are allowed.

```
Type        Example                    Bioinformatics use
--------    -----------------------    -----------------------------------
int         42                         Sequence length, read count
float       0.487                      GC content, E-value, p-value
str         "ATGCGA"                   DNA/RNA/protein sequences, gene names
bool        True / False               Is the sequence valid? Has a stop codon?
NoneType    None                       Missing data, function with no return
```

Use `type()` to check a value's type.

In [ ]:
# Check types of bioinformatics values
read_count = 15000000
gc_fraction = 0.52
organism = "Escherichia coli"
is_model_organism = True
annotation = None

values = [read_count, gc_fraction, organism, is_model_organism, annotation]
for v in values:
    print(f"{str(v):25s} -> {type(v).__name__}")

---

## 3. Integers (`int`)

Integers are whole numbers with no decimal point. In bioinformatics, you use them for counts, positions, and indices.

Python integers have **unlimited precision** -- they can be as large as your memory allows.

In [ ]:
# Typical bioinformatics integers
sequence_length = 3088286401          # human genome length in bp
num_genes = 20000                     # approximate protein-coding genes
read_depth = 30                       # sequencing coverage
chromosome_number = 23                # human haploid chromosome count

print(f"Human genome:  {sequence_length:,} bp")   # comma-separated formatting
print(f"Protein-coding genes: ~{num_genes:,}")
print(f"Target coverage: {read_depth}x")

In [ ]:
# Integers have unlimited precision in Python
huge_number = 2 ** 100
print(f"2^100 = {huge_number}")
print(f"Number of digits: {len(str(huge_number))}")
print(f"Type: {type(huge_number)}")

---

## 4. Floating-Point Numbers (`float`)

Floats represent decimal numbers. They are used for measurements, percentages, scores, and probabilities.

In [ ]:
# Typical bioinformatics floats
gc_content = 0.508                    # GC fraction
melting_temp = 72.3                   # PCR primer Tm in degrees C
e_value = 1.5e-42                     # BLAST E-value (scientific notation)
p_value = 0.0031                      # statistical significance

print(f"GC content: {gc_content}")
print(f"GC content as %: {gc_content * 100:.1f}%")
print(f"Melting temp: {melting_temp} C")
print(f"E-value: {e_value}")
print(f"E-value formatted: {e_value:.2e}")
print(f"p-value: {p_value}")

### Floating-point precision warning

Floats are stored in binary, which means some decimal numbers cannot be represented exactly. This is a fundamental limitation of floating-point arithmetic, not a Python bug.

In [ ]:
# Floating-point surprise
print(0.1 + 0.2)          # not exactly 0.3!
print(0.1 + 0.2 == 0.3)   # False!

# In practice, compare floats with a tolerance
result = 0.1 + 0.2
expected = 0.3
tolerance = 1e-9

print(f"Close enough? {abs(result - expected) < tolerance}")  # True

---

## 5. Strings (`str`)

Strings are sequences of characters. They are the **most important data type in bioinformatics** because DNA, RNA, and protein sequences are all represented as strings.

### Creating strings

In [ ]:
# Three ways to create strings
dna = "ATGCGATCGATCG"          # double quotes
rna = 'AUGCGAUCGAUCG'          # single quotes (identical behavior)
protein = """MKWVTFISLLLLFSSAYS"""  # triple quotes (can span multiple lines)

# Multi-line string (useful for FASTA headers, etc.)
fasta_entry = """>sp|P04637|P53_HUMAN Cellular tumor antigen p53
MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP
DEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYPQGLNGTVNLPGRNSFEV"""

print(fasta_entry[:80] + "...")

### String indexing

Each character in a string has an **index** (position number). Python uses zero-based indexing.

```
Index:    0   1   2   3   4   5   6   7
Seq:      A   T   G   C   G   A   T   C
Neg idx: -8  -7  -6  -5  -4  -3  -2  -1
```

In [ ]:
dna = "ATGCGATC"

# Positive indexing (from the left)
print(f"First nucleotide:  dna[0]  = {dna[0]}")
print(f"Third nucleotide:  dna[2]  = {dna[2]}")

# Negative indexing (from the right)
print(f"Last nucleotide:   dna[-1] = {dna[-1]}")
print(f"Second to last:    dna[-2] = {dna[-2]}")

### String slicing

Slicing extracts a substring. Syntax: `string[start:stop:step]`

- `start` is inclusive (default: 0)
- `stop` is exclusive (default: end of string)
- `step` is the increment (default: 1)

In [ ]:
dna = "ATGAAACCCGGGTAA"

# Extract the start codon (first 3 nucleotides)
start_codon = dna[0:3]
print(f"Start codon: {start_codon}")       # ATG

# Extract the stop codon (last 3 nucleotides)
stop_codon = dna[-3:]
print(f"Stop codon:  {stop_codon}")        # TAA

# Extract the coding region between start and stop
coding = dna[3:-3]
print(f"Coding region: {coding}")          # AAACCCGGG

# Every third nucleotide (first position of each codon)
first_positions = dna[0::3]
print(f"First codon positions: {first_positions}")  # AACGT

# Reverse the sequence
reversed_dna = dna[::-1]
print(f"Reversed: {reversed_dna}")         # AATGGGCCCAAAGTA

### String immutability

Strings in Python are **immutable** -- you cannot change individual characters. Instead, you create a new string.

In [ ]:
dna = "ATGCGA"

# This will raise a TypeError:
# dna[0] = "G"  # TypeError: 'str' object does not support item assignment

# Instead, create a new string
mutated = "G" + dna[1:]   # point mutation: A -> G at position 0
print(f"Original: {dna}")
print(f"Mutated:  {mutated}")

### Essential string methods for bioinformatics

Methods are functions that belong to a string. Call them with `string.method()`.

In [ ]:
dna = "atgcGATCgatc"

# Case conversion -- important when sequences come in mixed case
print(f"Upper: {dna.upper()}")       # ATGCGATCGATC
print(f"Lower: {dna.lower()}")       # atgcgatcgatc

In [ ]:
dna = "ATGCGATCGATCGTAGC"

# count() -- count occurrences of a substring
print(f"Number of G's: {dna.count('G')}")
print(f"Number of 'ATC' motifs: {dna.count('ATC')}")

In [ ]:
# find() -- find the position of a substring (-1 if not found)
dna = "ATGAAACCCGGGTAA"
print(f"Position of 'GGG': {dna.find('GGG')}")   # 9
print(f"Position of 'TTT': {dna.find('TTT')}")   # -1 (not found)

In [ ]:
# replace() -- replace all occurrences of a substring
dna = "ATGCGATCG"
rna = dna.replace("T", "U")    # DNA to RNA transcription
print(f"DNA: {dna}")
print(f"RNA: {rna}")

In [ ]:
# startswith() and endswith() -- check sequence boundaries
cds = "ATGAAACCCGGGTAA"
print(f"Starts with ATG (start codon)? {cds.startswith('ATG')}")
print(f"Ends with TAA (stop codon)?    {cds.endswith('TAA')}")

# Check for any stop codon
has_stop = cds.endswith(("TAA", "TAG", "TGA"))
print(f"Ends with any stop codon?      {has_stop}")

In [ ]:
# split() and join() -- essential for parsing biological file formats

# Parse a FASTA header
header = ">sp|P04637|P53_HUMAN Cellular tumor antigen p53"
parts = header.split("|")
print(f"Database: {parts[0][1:]}")
print(f"Accession: {parts[1]}")
print(f"Entry name: {parts[2].split()[0]}")

# Join codons with a separator
codons = ["ATG", "AAA", "CCC", "GGG", "TAA"]
formatted = " - ".join(codons)
print(f"\nCodons: {formatted}")

In [ ]:
# strip() -- remove whitespace (critical when reading files)
messy_line = "  ATGCGATCG  \n"
clean = messy_line.strip()
print(f"Before strip: '{messy_line}'")
print(f"After strip:  '{clean}'")

### The DNA complement with `str.maketrans()` and `translate()`

This is the most efficient way to compute a DNA complement in pure Python.

In [ ]:
dna = "ATGCGATCGATCGTAGC"

# Build a translation table: A<->T, G<->C
complement_table = str.maketrans("ATGC", "TACG")

complement = dna.translate(complement_table)
reverse_complement = complement[::-1]

print(f"Original:           5'-{dna}-3'")
print(f"Complement:         3'-{complement}-5'")
print(f"Reverse complement: 5'-{reverse_complement}-3'")

---

## 6. Booleans (`bool`)

Booleans have exactly two values: `True` and `False`. They are the result of comparisons and logical operations, and they drive `if` statements and `while` loops.

In [ ]:
# Booleans from comparisons
seq_length = 1500
print(f"Is length > 1000? {seq_length > 1000}")    # True
print(f"Is length == 1500? {seq_length == 1500}")   # True
print(f"Is length < 100? {seq_length < 100}")       # False

In [ ]:
# Sequence validation with booleans
sequence = "ATGCGATCGA"
valid_nucleotides = set("ATGC")

# Check if every character is a valid nucleotide
is_valid = all(nuc in valid_nucleotides for nuc in sequence.upper())
print(f"Sequence '{sequence}' is valid DNA: {is_valid}")

# Check a sequence with an invalid character
bad_sequence = "ATGCXYZ"
is_valid = all(nuc in valid_nucleotides for nuc in bad_sequence.upper())
print(f"Sequence '{bad_sequence}' is valid DNA: {is_valid}")

### Truthy and falsy values

Python treats certain non-boolean values as `True` or `False` in boolean contexts:

| Falsy (treated as False) | Truthy (treated as True) |
|--------------------------|-------------------------|
| `0`, `0.0` | Any non-zero number |
| `""` (empty string) | Any non-empty string |
| `[]` (empty list) | Any non-empty list |
| `None` | Almost everything else |

In [ ]:
# Practical use: check if a sequence is empty
sequence = ""
if not sequence:
    print("Warning: empty sequence!")

sequence = "ATGCGA"
if sequence:
    print(f"Sequence has {len(sequence)} nucleotides")

---

## 7. None

`None` represents the absence of a value. It is Python's version of "null" or "N/A".

In [ ]:
# Common uses of None

# 1. Indicate missing data
gene_function = None   # function not yet annotated

if gene_function is None:
    print("Gene function: unknown / not annotated")

# 2. Functions that do not return a value
result = print("hello")   # print() returns None
print(f"Return value of print(): {result}")

# Always use 'is' (not ==) to check for None
print(f"gene_function is None: {gene_function is None}")

---

## 8. Type Conversion

You often need to convert between types, especially when reading data from files (which always comes in as strings).

```
str(42)    -->  "42"       int   -> str
int("42")  -->  42         str   -> int
float(5)   -->  5.0        int   -> float
int(3.7)   -->  3          float -> int  (truncates!)
round(3.7) -->  4          float -> int  (rounds)
bool(1)    -->  True       int   -> bool
bool(0)    -->  False
```

In [ ]:
# Simulating reading a line from a data file
line = "BRCA1\t17\t43044295\t43170245\t0.423"
fields = line.split("\t")

gene = fields[0]                     # already a string
chrom = int(fields[1])               # str -> int
start = int(fields[2])               # str -> int
end = int(fields[3])                 # str -> int
gc = float(fields[4])               # str -> float

length = end - start
print(f"Gene: {gene}")
print(f"Location: chr{chrom}:{start}-{end}")
print(f"Length: {length:,} bp")
print(f"GC content: {gc * 100:.1f}%")

### Warning: `int()` truncates, `round()` rounds

In [ ]:
values = [3.2, 3.5, 3.7, -3.2, -3.7]

print(f"{'Value':>8} {'int()':>8} {'round()':>8}")
print("-" * 28)
for v in values:
    print(f"{v:>8.1f} {int(v):>8} {round(v):>8}")

---

## 9. Putting It Together: Bioinformatics Applications

### Application 1: GC Content Calculator

In [ ]:
def gc_content(sequence):
    """Calculate GC content as a percentage."""
    seq = sequence.upper()
    g = seq.count("G")
    c = seq.count("C")
    return (g + c) / len(seq) * 100

# Test with sequences from different organisms
test_sequences = {
    "Plasmodium falciparum (malaria)": "AATTATTTAATTAAATAAATTAATTAATTATTTAA",
    "Human TP53 exon 1":               "ATGGAGGAGCCGCAGTCAGATCCTAGCGTGAGTTTGC",
    "Thermus thermophilus (hot spring)": "GCGCCGGCGCGGCCGCGCCCGCCGCGGCGCCG",
}

for name, seq in test_sequences.items():
    print(f"{name:40s} GC = {gc_content(seq):.1f}%")

### Application 2: Molecular Weight Estimation

The molecular weight of a single-stranded DNA molecule can be estimated from its nucleotide composition.

| Nucleotide | Molecular weight (g/mol) |
|------------|-------------------------|
| dAMP | 331.2 |
| dTMP | 322.2 |
| dGMP | 347.2 |
| dCMP | 307.2 |

For each phosphodiester bond formed, one water molecule (18.02 g/mol) is lost.

In [ ]:
# Nucleotide molecular weights (monophosphate forms)
NUC_WEIGHTS = {"A": 331.2, "T": 322.2, "G": 347.2, "C": 307.2}
WATER = 18.02

def dna_molecular_weight(sequence):
    """Estimate molecular weight of a single-stranded DNA in Daltons."""
    seq = sequence.upper()
    total = 0.0
    for nuc in seq:
        total += NUC_WEIGHTS[nuc]
    # Subtract water for each phosphodiester bond (n-1 bonds)
    total -= (len(seq) - 1) * WATER
    return total

primer = "ATGCGATCGATCGATCGATCG"   # 21-mer primer
mw = dna_molecular_weight(primer)
print(f"Primer:  {primer}")
print(f"Length:  {len(primer)} nt")
print(f"MW:      {mw:,.1f} Da")
print(f"MW:      {mw/1000:.2f} kDa")

### Application 3: Sequence Report

In [ ]:
def sequence_report(sequence, name="Unknown"):
    """Print a comprehensive report for a DNA sequence."""
    seq = sequence.upper().strip()
    length = len(seq)

    # Validate
    invalid = set(seq) - set("ATGC")
    if invalid:
        print(f"ERROR: Invalid characters found: {invalid}")
        return

    # Nucleotide counts
    a = seq.count("A")
    t = seq.count("T")
    g = seq.count("G")
    c = seq.count("C")
    gc = (g + c) / length * 100

    print("=" * 50)
    print(f"  SEQUENCE REPORT: {name}")
    print("=" * 50)
    if length > 40:
        print(f"  Sequence: {seq[:20]}...{seq[-20:]}")
    else:
        print(f"  Sequence: {seq}")
    print(f"  Length:   {length} bp")
    print(f"  A: {a:>5} ({a/length*100:.1f}%)")
    print(f"  T: {t:>5} ({t/length*100:.1f}%)")
    print(f"  G: {g:>5} ({g/length*100:.1f}%)")
    print(f"  C: {c:>5} ({c/length*100:.1f}%)")
    print(f"  GC content: {gc:.2f}%")
    print(f"  Codons:     {length // 3} complete")
    print(f"  Starts with ATG: {seq.startswith('ATG')}")
    print(f"  Ends with stop:  {seq.endswith(('TAA', 'TAG', 'TGA'))}")
    print("=" * 50)

# Test
sequence_report("ATGAAACCCGGGTTTTCCCAAAGGGCCCGGGTAA", name="Test CDS")

---

## 10. Exercises

### Exercise 1: Nucleotide Counting

Count each nucleotide in the sequence below and print the results as a formatted table.

In [ ]:
sequence = "AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATTAAAAAAAGAGTGTCTGATAGCAGC"

# Your code here:
# Expected output: A: 20, C: 12, G: 17, T: 21


### Exercise 2: Molecular Weight Calculation

A protein has the amino acid composition shown below. Calculate its approximate molecular weight.

- Average amino acid mass: 110 Da
- Water lost per peptide bond: 18 Da
- Formula: MW = (n * 110) - (n-1) * 18, where n is the number of amino acids

In [ ]:
protein_sequence = "MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPS"
num_amino_acids = len(protein_sequence)

# Your code here:
# Calculate and print the approximate molecular weight in Da and kDa


### Exercise 3: Codon Extraction

Extract all codons (triplets) from the sequence below and print them.

In [ ]:
cds = "ATGGCCGATCGATAG"

# Your code here:
# Expected output: ATG, GCC, GAT, CGA, TAG
# Hint: use slicing with step 3 in a loop or list comprehension


### Exercise 4: Reverse Complement

Write code that computes the reverse complement of a DNA sequence. Test it on the sequence below.

Base pairing: A <-> T, G <-> C

In [ ]:
dna = "AAAACCCGGT"

# Your code here:
# Expected output: ACCGGGTTTT
# Hint: use str.maketrans() and translate(), then reverse with [::-1]


### Exercise 5: Parse a FASTA Header

Extract the accession number, entry name, and organism from the UniProt FASTA header below.

In [ ]:
header = ">sp|P04637|P53_HUMAN Cellular tumor antigen p53 OS=Homo sapiens OX=9606 GN=TP53 PE=1 SV=4"

# Your code here:
# Expected:
#   Accession: P04637
#   Entry name: P53_HUMAN
#   Gene name: TP53
#
# Hint: use split("|"), then further split or find "GN=" in the description


---

## Summary

| Type | Example | Key bioinformatics use |
|------|---------|------------------------|
| `int` | `1500` | Sequence lengths, positions, read counts |
| `float` | `0.487` | GC content, E-values, p-values |
| `str` | `"ATGCGA"` | DNA/RNA/protein sequences, gene names |
| `bool` | `True` | Sequence validation, filter conditions |
| `None` | `None` | Missing annotations, no return value |

**Key takeaways**:

1. Use descriptive `snake_case` variable names
2. Strings are immutable sequences of characters -- perfect for biological sequences
3. Master string slicing (`seq[start:stop:step]`) early -- you will use it constantly
4. Know the difference between `int()` (truncation) and `round()` (rounding)
5. Always convert strings to numbers when reading data from files

### Next module: Operators and Expressions

We will learn all the operators Python provides and use them for biological calculations.

---[< Previous: Variables + Data Types: Overview](01_variables_and_data_types.ipynb) | [Tier 1: Python for Bioinformatics Overview](../README.md) | [Next: Operators >](../03_Operators_and_Expressions/01_operators.ipynb)---